# Manufacturing Quality Perceptron With NumPy

This notebook repeats the quality-control classifier using NumPy arrays and a batch perceptron. All parts are scored and used to update the model together, making the training operations compact and fully vectorised.

## 1. A dot product is a weighted sum

For two features, `np.dot(weights, features)` performs the same multiplication and addition that we wrote manually.

In [ ]:
import numpy as np

example_weights = np.array([-0.4, -0.2])
example_features = np.array([0.25, 1.0])
manual_sum = example_weights[0] * example_features[0] + example_weights[1] * example_features[1]
dot_sum = np.dot(example_weights, example_features)

print(f"Manual weighted sum: {manual_sum:.3f}")
print(f"NumPy dot product:   {dot_sum:.3f}")
assert np.isclose(manual_sum, dot_sum)

## 2. Initialize the perceptron

The feature order is `[dimension_error_mm, surface_defects]`. Labels remain **ACCEPT = 1** and **REJECT = 0**. This is synthetic teaching data, not a production quality-control system.

In [ ]:
np.random.seed(42)
weights = np.random.uniform(-1, 1, size=2)
bias = float(np.random.uniform(-1, 1))
learning_rate = 0.1
max_epochs = 100

print("Starting weights:", np.round(weights, 3))
print(f"Starting bias: {bias:.3f}")

## 3. Store measurements as NumPy arrays

The feature matrix `X` has one row per part and one column per feature. Its shape is `(12, 2)`: 12 parts by 2 features. The label vector `y` has shape `(12,)`, with one label per part.

In [ ]:
part_ids = np.array([
    "P01", "P02", "P03", "P04", "P05", "P06",
    "P07", "P08", "P09", "P10", "P11", "P12",
])

X = np.array([
    [0.10, 0.0],
    [0.15, 1.0],
    [0.20, 0.0],
    [0.25, 1.0],
    [0.30, 0.0],
    [0.05, 1.0],
    [0.55, 0.0],
    [0.45, 2.0],
    [0.20, 3.0],
    [0.70, 1.0],
    [0.35, 2.0],
    [0.60, 2.0],
])
y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

test_part_ids = np.array(["T01", "T02", "T03", "T04"])
X_test = np.array([
    [0.12, 0.0],
    [0.28, 1.0],
    [0.50, 1.0],
    [0.10, 3.0],
])
y_test = np.array([1, 1, 0, 0])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Loaded {len(part_ids)} training parts and {len(test_part_ids)} test parts.")

## 4. Train with vector operations

This is the **batch perceptron** variant. Unlike the sequential updates in the from-scratch notebook, every part contributes to one update per epoch.

- `X @ weights` produces all 12 scores at once.
- Applying the threshold produces all 12 predictions.
- `X.T @ errors` combines the corrections into one update for both weights.
- The epoch loop remains because the model may need several rounds of corrections before it classifies every part correctly.

In [ ]:
history = []
for epoch in range(max_epochs):
    scores = X @ weights + bias
    predictions = (scores >= 0).astype(int)
    errors = y - predictions
    error_count = np.count_nonzero(errors)

    history.append(int(error_count))
    print(f"Epoch {epoch + 1:3d} | Errors: {error_count}")
    if error_count == 0:
        print(f"Training complete after {epoch + 1} epoch(s).")
        break

    weights += learning_rate * (X.T @ errors)
    bias += learning_rate * errors.sum()
else:
    raise RuntimeError("The perceptron did not converge within 100 epochs.")

## 5. Predict unseen parts

The matrix operation calculates every test score and prediction together. The small loop below only formats one readable output line per part; it does not train or calculate predictions.

In [ ]:
test_scores = X_test @ weights + bias
test_predictions = (test_scores >= 0).astype(int)

assert np.array_equal(test_predictions, y_test)

for part_id, score, prediction, expected in zip(
    test_part_ids, test_scores, test_predictions, y_test
):
    label = "ACCEPT" if prediction == 1 else "REJECT"
    expected_label = "ACCEPT" if expected == 1 else "REJECT"
    print(f"{part_id}: score={score:+.3f} -> {label} (expected {expected_label})")

## 6. Connect the compact notation to the arithmetic

Use T01 to verify one more time that the dot product is only shorthand for multiply-and-add. Negative learned weights mean larger measurement values push the score toward `REJECT`.

In [ ]:
t01_features = X_test[0]
manual_score = (
    weights[0] * t01_features[0]
    + weights[1] * t01_features[1]
    + bias
)
dot_score = np.dot(weights, t01_features) + bias

print(f"Manual score: {manual_score:+.3f}")
print(f"Dot-product score: {dot_score:+.3f}")
print("Learned weights:", np.round(weights, 3))
print(f"Bias: {bias:+.3f}")
assert np.isclose(manual_score, dot_score)
assert np.all(weights < 0)